# Bond Pricing and Yield Curves
In this notebook, I will be using Python to explore bond pricing and yield curve construction. I will begin by building a pricing engine to price bonds, and then use this to solve the yield to maturity for a given price. From there, I will calculate some key risk measurements for fixed income like duration and convexity. I will then use this to approximate a price change for a given yield shift.

On the yield curve side, I will take Australian Government Bond (AGB) data from the Australian Office of Financial Managemement (AOFM) and use it to build a yield curve using bootstrapping. From this, I will also calculate a forward rates curve, and also use it to price bonds of the curve. 

In [2]:
# Importing packages

import numpy as np
import pandas as pd
import matplotlib as plt

# Bond Pricing
First I will begin by creating a bond pricing engine. For some given cash flows/coupon, yield, face value and term to maturity, the price of a bond will be calculated. The following forumla will be used to calculate price:

$$ P = \sum_{t = 1}^{N} \frac{C_t}{(1+r)^t} + \frac{F}{(1+r)^N}. $$

Where:
* $F$ is the face value of the bond, or the amount the bond pays until maturity
* $C_t$ is the coupon payment at time period t, where $C_t = F * i$ and $i$ is the annual coupon rate
* $n$ is the total number of years to maturity
* $r$ is the required discount rate or yield to maturity

The above formula is for annual compounding periods. For a lot of bonds, coupon periods are more frequent than annually, for example, AGBs compounded semi-annually. The formula for bonds that are compounded more frequently can be adjusted to the following, where $m$ is the number of compounding periods per year.:

$$  P = \sum_{t = 1}^{m \cdot N} \frac{C/m}{(1+r/m)^t} + \frac{F}{(1+r/m)^{m \cdot N}}. $$

For a zero-coupon bond, the formula price automatically becomes the discounted face value of the bond (no need for an additional formula in our code). 

In [5]:
def bond_pricer(face_value, annual_coupon_rate, discount_rate, years_to_maturity, compounding_periods):
  # Adjust the compounding periods to at least 1 to prevent division by zero
  if compounding_periods <= 0:
    compounding_periods = 1
  
  # Calculate the discounted face value
  discounted_face_value = face_value / (1 + discount_rate / compounding_periods)**(years_to_maturity * compounding_periods)
  
  # Calculate the discounted cash flows (coupons)
  num_payments = years_to_maturity * compounding_periods # calculate the number of coupon payments
  coupon_payment = (face_value * annual_coupon_rate) / compounding_periods # calculate the coupon payment for each compounding period/payment
  
  discounted_coupons = np.zeros(num_payments) # initialise array for discounted coupons for each payment over the life of the bond
  
  for payment in range(num_payments):
    discounted_coupons[payment] = coupon_payment / (1 + discount_rate / compounding_periods)**(payment)
    
  # Calculate the total price of the bond by summing the the discounted cash flows the to discounted face value, paid at maturity
  price = np.sum(discounted_coupons) + discounted_face_value
  
  # Return price
  return price

For example, a bond with a face value of \$100, an annual coupon rate of 5% with a market discount rate/yield to maturity of 3% for a 10 year bond and semi-annual compounding, we calculate the bond should be priced at \$117.81 (to two decimal places)

In [6]:
# Outlining the features of the bond
face_value = 100
annual_coupon_rate = 0.05
discount_rate = 0.03
years_to_maturity = 10
compounding_periods = 2

# Calculating the price
price = bond_pricer(face_value, annual_coupon_rate, discount_rate, years_to_maturity, compounding_periods)
print(f"Price: ${price:.2f}")


Price: $117.81


## Yield to maturity
We can calculate the yield to maturity of a bond by observing the bond's market price, then backing out the yield to maturity, which we will do in this section of the notebook. 

It is important to not ethat when calculating the theoretical price of a bond, an appropriate discount rate is selected, called the required rate of return. This is calculated as the sum of the following components:

$$ Discount Rate = Risk-Free Rate + Default Risk Premium + Liquidity Premium + Term Premium. $$

* The risk-free rate is the yield of a government bond with the exact same maturity.
* Credit risk is the extra yield to compensate investors for the default risk of a company, This can be caulculated by checking the bond issuer's credit rating, then using relevant credit spreads.
* Liquidity premium is the extra yield if the bond is difficult to sell quickly/convert to cash without impacting the market price. 
* Term premium (also known as interest rate risk premium) is the additional compensation required for investing money in a bond, exposing inverstors to future interest rater fluctuations. This can already be considered in the risk-free yield curve.

Now with all of that said, we can begin to find the yield of maturity given a price observed in the market. To do this, we will use the scipy solver function, and will apply this to both a hypotheitcal example and a real example of finding the yield of an AGB. 